# Baseline Benchmark for Sarcasm Style Rewriting

This notebook benchmarks five baseline models on an uploaded benchmark dataset and compares them on two metrics already implemented in this repo:

- cosine similarity via `evaluation/text_similarity.py`
- perplexity via `evaluation/text_perplexity.py`

Upload one benchmark dataset JSON file at runtime. The notebook uses only `headline` and `is_sarcastic` from that file, builds rewrite prompts from `headline`, and asks each model to:

- rewrite sarcastic headlines (`is_sarcastic = 1`) into HuffPost-style non-sarcastic headlines
- rewrite non-sarcastic headlines (`is_sarcastic = 0`) into Onion-style sarcastic headlines

The benchmarked models are:

- `google/flan-t5-base`
- `facebook/bart-base`
- `t5-base`
- `EleutherAI/gpt-neo-1.3B`
- `gpt2-large`


In [ ]:
# Pull code, install dependencies

import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/LINCHENYU2030S/CS4248_group_project.git"
REPO_ROOT = Path("/content/CS4248_group_project")
LOCAL_PROJECT_ROOT = Path("/content/CS4248_group_project")
LOCAL_PROJECT_EVAL_DIR = LOCAL_PROJECT_ROOT / "evaluation"
LOCAL_EVAL_DIR = Path("/content/evaluation")

current_dir = Path.cwd().resolve()
if current_dir.name == "evaluation":
    project_eval_dir = current_dir
elif current_dir.name == "notebooks" and (current_dir.parent / "evaluation").exists():
    project_eval_dir = current_dir.parent / "evaluation"
elif (current_dir / "evaluation").exists():
    project_eval_dir = current_dir / "evaluation"
elif LOCAL_PROJECT_EVAL_DIR.exists():
    project_eval_dir = LOCAL_PROJECT_EVAL_DIR
elif LOCAL_EVAL_DIR.exists():
    project_eval_dir = LOCAL_EVAL_DIR
elif "google.colab" in sys.modules:
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    project_eval_dir = REPO_ROOT / "evaluation"
else:
    raise RuntimeError(
        f"Could not locate the repo evaluation directory from: {current_dir}"
    )

os.chdir(project_eval_dir)
PROJECT_EVAL_DIR = Path.cwd().resolve()
PROJECT_ROOT = PROJECT_EVAL_DIR.parent
if PROJECT_EVAL_DIR.name != "evaluation":
    raise RuntimeError(f"Expected to run inside the evaluation directory, got: {PROJECT_EVAL_DIR}")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
for path in [PROJECT_ROOT, PROJECT_EVAL_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "transformers>=4.39.0",
        "sentence-transformers>=2.6.0",
        "accelerate>=0.28.0",
        "seaborn>=0.13.0",
        "ipywidgets>=8.1.0",
        "sentencepiece>=0.1.99",
    ],
    check=True,
)

print(f"Working directory: {PROJECT_EVAL_DIR}")
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
# Upload benchmark dataset at run time

if "google.colab" not in sys.modules:
    raise RuntimeError("This notebook is set up for Colab upload flow. Run it in Colab and upload one JSON dataset file.")

from google.colab import files

print("Upload exactly one JSON benchmark dataset file.")
uploaded = files.upload()

json_candidates = [name for name in uploaded if name.lower().endswith(".json")]
if len(json_candidates) != 1:
    raise FileNotFoundError(
        "Expected exactly one uploaded JSON file. "
        f"Uploaded files: {list(uploaded)}"
    )

DATASET_FILENAME = json_candidates[0]
DATASET_PATH = PROJECT_EVAL_DIR / DATASET_FILENAME
DATASET_PATH.write_bytes(uploaded[DATASET_FILENAME])

print(f"Dataset saved to: {DATASET_PATH}")


In [ ]:
# Set seed, set up device, and configure plotting

import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from matplotlib import pyplot as plt
from transformers import set_seed

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_colwidth", 120)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE != "cuda":
    print("Warning: GPU is not enabled. This benchmark is intended for a Colab T4, so CPU will be much slower.")


In [ ]:
# Define models and benchmark parameters

@dataclass(frozen=True)
class ModelSpec:
    key: str
    label: str
    hf_name: str
    architecture: str
    batch_size: int
    max_source_length: int = 96
    max_new_tokens: int = 40


MODEL_SPECS = [
    ModelSpec(
        key="flan_t5_base",
        label="FLAN-T5-base",
        hf_name="google/flan-t5-base",
        architecture="seq2seq",
        batch_size=8,
    ),
    ModelSpec(
        key="bart_base",
        label="BART-base",
        hf_name="facebook/bart-base",
        architecture="seq2seq",
        batch_size=8,
    ),
    ModelSpec(
        key="t5_base",
        label="T5-base",
        hf_name="t5-base",
        architecture="seq2seq",
        batch_size=8,
    ),
    ModelSpec(
        key="gpt_neo_1_3b",
        label="GPT-Neo-1.3B",
        hf_name="EleutherAI/gpt-neo-1.3B",
        architecture="causal",
        batch_size=4,
    ),
    ModelSpec(
        key="gpt2_large",
        label="GPT-2-large",
        hf_name="gpt2-large",
        architecture="causal",
        batch_size=4,
    ),
]

OUTPUT_STEM = Path(DATASET_FILENAME).stem
OUTPUT_DIR = PROJECT_EVAL_DIR / "baseline_outputs" / OUTPUT_STEM
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_ONLY_MODEL_KEYS = None
FORCE_REGENERATE = True  # Switch back to False after you have a clean cached run.
FORCE_RESCORE = True
USE_FP16_ON_GPU = True
PERPLEXITY_BATCH_SIZE = 8

COMMON_GENERATION_KWARGS = {
    "do_sample": False,
    "num_beams": 1,
    "min_new_tokens": 4,
    "repetition_penalty": 1.05,
}

RUN_NAME = f"{OUTPUT_STEM}_benchmark"
ACTIVE_MODEL_SPECS = MODEL_SPECS
if RUN_ONLY_MODEL_KEYS is not None:
    ACTIVE_MODEL_SPECS = [spec for spec in MODEL_SPECS if spec.key in set(RUN_ONLY_MODEL_KEYS)]

print("Models:", [spec.hf_name for spec in ACTIVE_MODEL_SPECS])
print(f"Dataset path: {DATASET_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Run name: {RUN_NAME}")


In [ ]:
# Helper functions imported from evaluation/utils.py

from evaluation.utils import (
    evaluate_generations,
    load_dataset,
    run_generation_for_model,
    summarise_results,
)


In [ ]:
dataset_df = load_dataset(dataset_path=DATASET_PATH)

display(dataset_df.head())
print(f"Number of benchmark examples: {len(dataset_df):,}")
print("Label counts:")
print(dataset_df["is_sarcastic"].value_counts().sort_index())


In [ ]:
all_results = []

for model_spec in ACTIVE_MODEL_SPECS:
    print(f"\n=== Running benchmark generation for {model_spec.label} ===")
    generation_kwargs = {
        **COMMON_GENERATION_KWARGS,
        "max_new_tokens": model_spec.max_new_tokens,
    }

    generated_df = run_generation_for_model(
        dataset_df=dataset_df,
        model_spec=model_spec,
        output_dir=OUTPUT_DIR,
        run_name=RUN_NAME,
        device=DEVICE,
        batch_size=model_spec.batch_size,
        max_source_length=model_spec.max_source_length,
        generation_kwargs=generation_kwargs,
        use_fp16_on_gpu=USE_FP16_ON_GPU,
        force_regenerate=FORCE_REGENERATE,
    )

    print(f"Scoring {model_spec.label} outputs")
    scored_df = evaluate_generations(
        generated_df=generated_df,
        output_dir=OUTPUT_DIR,
        run_name=RUN_NAME,
        perplexity_batch_size=PERPLEXITY_BATCH_SIZE,
        force_rescore=FORCE_RESCORE,
    )
    all_results.append(scored_df)

benchmark_results_df = pd.concat(all_results, ignore_index=True)
benchmark_results_path = OUTPUT_DIR / f"{RUN_NAME}_all_model_metrics.csv"
benchmark_results_df.to_csv(benchmark_results_path, index=False)

print(f"Saved combined benchmark results to {benchmark_results_path}")
display(benchmark_results_df.head())


In [ ]:
summary_df = summarise_results(benchmark_results_df)
summary_path = OUTPUT_DIR / f"{RUN_NAME}_summary.csv"
summary_df.to_csv(summary_path, index=False)

display(summary_df)
print(f"Saved summary metrics to {summary_path}")


In [ ]:
plot_df = summary_df.copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.barplot(
    data=plot_df,
    x="model_label",
    y="mean_cosine_similarity",
    hue="model_label",
    ax=axes[0],
    palette="crest",
)
if axes[0].legend_ is not None:
    axes[0].legend_.remove()
axes[0].set_title("Meaning Preservation")
axes[0].set_xlabel("")
axes[0].set_ylabel("Mean cosine similarity")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(
    data=plot_df,
    x="model_label",
    y="mean_perplexity",
    hue="model_label",
    ax=axes[1],
    palette="flare",
)
if axes[1].legend_ is not None:
    axes[1].legend_.remove()
axes[1].set_title("Fluency")
axes[1].set_xlabel("")
axes[1].set_ylabel("Mean perplexity (lower is better)")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


In [ ]:
valid_results_df = benchmark_results_df.loc[
    benchmark_results_df["generated_headline"].fillna("").str.strip().ne("")
].copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.boxplot(
    data=valid_results_df,
    x="model_label",
    y="cosine_similarity",
    hue="model_label",
    ax=axes[0],
    palette="crest",
)
if axes[0].legend_ is not None:
    axes[0].legend_.remove()
axes[0].set_title("Cosine Similarity Distribution")
axes[0].set_xlabel("")
axes[0].set_ylabel("Cosine similarity")
axes[0].tick_params(axis="x", rotation=20)

sns.boxplot(
    data=valid_results_df,
    x="model_label",
    y="perplexity",
    hue="model_label",
    ax=axes[1],
    palette="flare",
    showfliers=False,
)
if axes[1].legend_ is not None:
    axes[1].legend_.remove()
axes[1].set_title("Perplexity Distribution")
axes[1].set_xlabel("")
axes[1].set_ylabel("Perplexity")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


In [ ]:
preview_rows = []
for model_label, frame in benchmark_results_df.groupby("model_label"):
    preview_rows.append(frame.sample(n=min(5, len(frame)), random_state=SEED))

preview_df = pd.concat(preview_rows, ignore_index=True)[[
    "model_label",
    "model_architecture",
    "source_style",
    "target_style",
    "target_publication",
    "headline",
    "generated_headline",
    "cosine_similarity",
    "perplexity",
]]

display(preview_df)
